In [1]:
# imports required 
import torch as t
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import random
import time
import math

import pandas as pd

In [2]:
dataframe=pd.read_csv("baby_names.csv")
names=dataframe['Name']
print(names.__len__())
print(names.head())

890627
0      Mary
1     Annie
2    Mattie
3      Ruby
4    Willie
Name: Name, dtype: object


In [3]:
#character vocabulary set up

allchars=sorted(set("".join(names)))
char_to_idx={ch:i for i,ch in enumerate(allchars)}
idx_to_char={i:ch for ch,i in char_to_idx.items()}

vocab_size=len(allchars)
print(vocab_size)

52


In [4]:
# in pytorch for rnn,lstm there is a expected shape of (seq_len,batch_size,vocab_size)
def name_to_tensor(name):
        new_ten=t.zeros(len(name),1,vocab_size)
        for i,j in enumerate(name):
                new_ten[i][0][char_to_idx[j]]=1
        return new_ten

In [5]:
class CharRNN(nn.Module):
        def __init__(self,input_size,hidden_size,output_size):
                super(CharRNN,self).__init__()
                self.hidden_size=hidden_size
                self.rnn_cell=nn.RNNCell(input_size,hidden_size)
                self.fc=nn.Linear(hidden_size,output_size)
        def forward(self,input_tensor,hidden):
                outputs=[]
                for i in range(input_tensor.size(0)):
                        hidden=self.rnn_cell(input_tensor[i],hidden)
                        output=self.fc(hidden)
                        outputs.append(output)
                return t.stack(outputs),hidden
        
        def init_hidden(self):
                return t.zeros(1,self.hidden_size)
        
        


In [10]:
def train(model,name_tensor,target_tensor,criterion,optimizer):
        hidden=model.init_hidden()
        model.zero_grad()
        output,hidden=model(name_tensor,hidden)
        loss=criterion(output.view(-1,vocab_size),target_tensor.view(-1))
        loss.backward()
        optimizer.step()
        return loss.item()


In [11]:
def target_from_name(name):
        target_ix=[char_to_idx[ch] for ch in name[1:]]
        target_ix.append(char_to_idx[name[-1]])
        return t.tensor(target_ix,dtype=t.long)


In [12]:
hidden_size=128
model=CharRNN(vocab_size,hidden_size,vocab_size)
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(),lr=0.05)


In [13]:
n_iters=1000

print_every=100
for iter in range(1,n_iters+1):
        name=random.choice(names)
        name_tensor=name_to_tensor(name)
        target_tensor=target_from_name(name)
        loss=train(model,name_tensor,target_tensor,criterion,optimizer)
        if iter%print_every==0:
                print(f"Iter {iter} Loss : {loss:.4f}")
                

Iter 100 Loss : 2.6766
Iter 200 Loss : 4.3534
Iter 300 Loss : 2.1494
Iter 400 Loss : 4.3293
Iter 500 Loss : 3.3927
Iter 600 Loss : 11.3688
Iter 700 Loss : 6.5298
Iter 800 Loss : 4.2296
Iter 900 Loss : 11.3690
Iter 1000 Loss : 6.5550


In [16]:
def sample(model,start_letter="A",max_length=20,temperature=0.8):
        with t.no_grad():
                input=name_to_tensor(start_letter)
                hidden=model.init_hidden()
                output_name=start_letter
                for i in range(max_length):
                        output,hidden=model(input,hidden)
                        top_i=output[-1].argmax(dim=1).item()
                        predicted_char=idx_to_char[top_i]
                        output_name+=predicted_char
                        input=name_to_tensor(predicted_char)
                return output_name
                

In [17]:
for _ in range(5):
        print(sample(model,start_letter=random.choice(allchars)))

nmaaaaaaaaaaaaaaaaaaa
mmaaaaaaaaaaaaaaaaaaa
vmaaaaaaaaaaaaaaaaaaa
amaaaaaaaaaaaaaaaaaaa
hmaaaaaaaaaaaaaaaaaaa
